# Sensor Data Quality Audit and ETL Pipeline

**Topic 19 — AAI-2606M**

Amina Tuzelova · Alua Nurlybekova · Zhanerke Pernesh

This notebook walks through the same Python project as `main.py`. It loads the included real outputs and lets you inspect the audit. It does not duplicate or change the original data.

## 1. Environment

Locally: after cloning from GitHub and installing `requirements.txt`, run `python download.py`, then `python main.py --run` from the project folder before running the analytical cells below. Use the same Python environment for the terminal commands and this notebook.

`data/raw/*.csv` and `data/processed/` are excluded from Git. This notebook reads the prepared source and processed files; a copy obtained through `git clone` does not include them.

If you have the complete `NK_Oil_Project_AAI2606M.zip` with raw and processed data, extract it and open this notebook from the project folder after installing `requirements.txt`. This full ZIP differs from a copy obtained through `git clone`.

In Google Colab, upload the complete `NK_Oil_Project_AAI2606M.zip` into `/content`, unzip it, and install dependencies. Uncomment the commands below only in Colab. The full project includes raw and processed data.


In [ ]:
# Google Colab only:
# !unzip -q NK_Oil_Project_AAI2606M.zip
# %cd /content/nk_oil_project
# %pip install -q -r requirements.txt

from pathlib import Path
import os
ROOT = Path.cwd()
if not (ROOT / "main.py").exists() and (ROOT / "nk_oil_project" / "main.py").exists():
    ROOT = ROOT / "nk_oil_project"
    os.chdir(ROOT)
assert (ROOT / "main.py").exists(), "Open this notebook inside the project folder."
print(ROOT)


## 2. Reproduce or inspect

The included outputs already come from the real data. The next two subprocess commands are optional only when the required raw CSVs and processed files already exist. After a fresh clone, prepare them with `python download.py` and `python main.py --run` from the project folder, or uncomment and execute the two commands below before running the analytical cells. A rerun updates benchmark times, which vary by machine; update the report consistently if you use those new times.


In [ ]:
import subprocess, sys
# subprocess.run([sys.executable, "download.py"], check=True)
# subprocess.run([sys.executable, "main.py", "--run"], check=True)

import pandas as pd
import json
from IPython.display import display, Image, HTML
summary = json.loads((ROOT / "outputs/summary.json").read_text(encoding="utf-8"))
df = pd.read_parquet(ROOT / "data/processed/observations.parquet")
print("Shape:", df.shape)
display(df.head())


## 3. Missing cells versus absent measurements

The existing rows contain no missing numeric values. A missing hour is an absent row. The valid grid contains 43 well–parameter pairs, each with 6,552 expected hours. NK-76 casing pressure is not applicable.


In [ ]:
coverage = pd.read_csv(ROOT / "outputs/coverage_by_series.csv")
print("Numeric nulls in existing rows:", df["value"].isna().sum())
print("Expected:", coverage.expected_hours.sum())
print("Observed:", coverage.observed_hours.sum())
print("Missing:", coverage.missing_hours.sum())
display(coverage.sort_values("missing_percent", ascending=False).head(10))


## 4. Long and wide format

Long format: one parameter per row. Wide format: parameters become columns. In the wide table, a null can mean an absent hour or a structurally inapplicable sensor. The completeness denominator uses the explicit applicable-pair table.


In [ ]:
wide = pd.read_parquet(ROOT / "data/processed/wide.parquet")
print("Wide shape:", wide.shape)
display(wide.loc[wide.well_id.eq("NK-7")].head())


## 5. SQL LAG and boundary gaps

`LAG` retrieves the preceding timestamp within the same well and parameter. An interval greater than one hour contains missing readings. A separate audit handles leading and trailing gaps.


In [ ]:
import sqlite3
from contextlib import closing
query = (ROOT / "sql/04_interior_gaps_lag.sql").read_text()
print(query)
with closing(sqlite3.connect(ROOT / "data/processed/wells.db")) as connection:
    gaps = pd.read_sql_query(query, connection)
print("Interior gaps:", len(gaps))
print("Missing observations inside gaps:", gaps.missing_hours.sum())
display(gaps.sort_values("missing_hours", ascending=False).head(10))
display(pd.read_csv(ROOT / "outputs/gaps_boundary.csv"))


## 6. Source flags and version reconciliation

Flags describe source quality concerns, not verified equipment failures. The source derives flags before averaging raw records into hours. Version comparison uses identical keys and a rounding-aware absolute tolerance of 0.000051.


In [ ]:
display(pd.read_csv(ROOT / "outputs/sql/05_individual_flag_counts.csv"))
display(pd.DataFrame([summary["v1_v2"]]).drop(columns=["v1_validation"]))
display(pd.read_csv(ROOT / "outputs/v1_v2_changed_examples.csv").head(5))


## 7. Baseline and two approaches

Python loop, Pandas groupby+shift and SQLite LAG solve the same interior-gap problem. Timing uses prepared inputs and median of three runs. Memory is Python-traced incremental allocation, not total RSS; SQLite native memory is not fully measured. Exact output equality is not a fault-prediction accuracy score.


In [ ]:
display(pd.read_csv(ROOT / "outputs/benchmark.csv"))
print(summary["benchmark_notes"]["timing_scope"])
print(summary["benchmark_notes"]["memory_scope"])


## 8. Visual evidence

The charts use observed readings without imputation. The area scale shift and NK-7 regime shift are annotated but not automatically corrected.


In [ ]:
display(Image(filename=str(ROOT / "outputs/figures/01_missing_measurements.png"), width=1000))
display(Image(filename=str(ROOT / "outputs/figures/04_nk7_regime.png"), width=1000))
# In Colab/Jupyter this embeds the interactive Plotly chart:
display(HTML((ROOT / "outputs/interactive_spm.html").read_text(encoding="utf-8")))


## 9. Conclusions to explain

- Coverage is 91.71%; missing observations are absent rows rather than numeric nulls.
- Source flags, statistical outliers and known change annotations have different meanings.
- All three gap algorithms agree. Pandas is fastest in the recorded prepared-input run.
- Version 2 changes many overlapping values as well as extending the time horizon.
- No verified equipment-failure labels exist, so this project does not claim failure prediction.

See [the dashboard demonstration guide](docs/DASHBOARD_GUIDE_RU.md) for short Russian explanations and `DATA_SOURCES.md` for attribution.
